In [3]:
# Notebook display tweaks (run once)
from IPython.display import HTML, display

display(HTML("""
<style>
  /* Make markdown headings smaller */
  .jp-Notebook .jp-MarkdownCell h1 { font-size: 1.6em; }
  .jp-Notebook .jp-MarkdownCell h2 { font-size: 1.25em; }
  .jp-Notebook .jp-MarkdownCell h3 { font-size: 1.10em; }
  .jp-Notebook .jp-MarkdownCell h4 { font-size: 1.00em; }
  /* Also slightly reduce overall markdown text */
  .jp-Notebook .jp-MarkdownCell { font-size: 0.95em; }
</style>
"""))

In [4]:
import sys, platform
print("sys.executable:", sys.executable)
print("python:", sys.version)
print("platform:", platform.platform())

sys.executable: c:\Git\APS-IFC\.venv\Scripts\python.exe
python: 3.14.0 (tags/v3.14.0:ebf955d, Oct  7 2025, 10:15:03) [MSC v.1944 64 bit (AMD64)]
platform: Windows-11-10.0.26100-SP0


#### 1) Choose the input JSON file
Set `input_path` to the model JSON you want to analyze, then run the cells below.

In [5]:
from pathlib import Path
import json

# Change this path if needed
input_path = Path(r"c:\Git\APS-IFC\JSON Whole Model\ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json")
assert input_path.exists(), input_path
input_path

WindowsPath('c:/Git/APS-IFC/JSON Whole Model/ASTIDC-STAN-HE-EPD-MVB1-M-E-0002.json')

In [6]:
data = json.loads(input_path.read_text(encoding="utf-8"))
assert isinstance(data, list), f"Expected list, got {type(data).__name__}"
len(data)

180

In [20]:
def safe_properties_count(element: dict) -> int:
    properties = element.get("Properties")
    if isinstance(properties, list):
        return len(properties)
    return 0


def safe_category_count(element: dict) -> int:
    """Counts categories (non-unique).
    
    IMPORTANT: This returns the number of property entries, even if a property has no `category`,
    so it matches `safe_properties_count` for the object.
    """
    return safe_properties_count(element)


def safe_distinct_category_count(element: dict) -> int:
    """Counts distinct non-empty category strings across Properties items."""
    properties = element.get("Properties")
    if not isinstance(properties, list):
        return 0
    categories = set()
    for item in properties:
        if not isinstance(item, dict):
            continue
        category = item.get("category")
        if category is None:
            continue
        text = str(category).strip()
        if not text:
            continue
        categories.add(text)
    return len(categories)

In [21]:
rows = []
row_no = 0
skipped_non_dict = 0
skipped_missing_name = 0

for element in data:
    if not isinstance(element, dict):
        skipped_non_dict += 1
        continue
    name = element.get("Name")
    if name is None:
        skipped_missing_name += 1
        continue
    row_no += 1
    rows.append({
        "No": row_no,
        "Name": name,
        "PropertiesCount": safe_properties_count(element),
        "UniqueCategoryCount": safe_distinct_category_count(element),
        "CategoryCount": safe_category_count(element),
    })

print("Total elements in JSON:\t", len(data))
print("Rows with Name (printed):\t", len(rows))
print("Skipped (not a dict):\t", skipped_non_dict)
print("Skipped (Name is null):\t", skipped_missing_name)

Total elements in JSON:	 180
Rows with Name (printed):	 180
Skipped (not a dict):	 0
Skipped (Name is null):	 0


In [23]:
# Print TSV (Excel-friendly)

# NOTE: This cell expects `rows` to be created by the previous cell (the one that loops over `data`).
# If you run this cell before running the earlier ones, `rows` won't exist yet.
if "rows" not in globals():
    if "data" in globals() and isinstance(data, list):
        # Rebuild rows from `data` if possible (in case earlier cells weren't run in order).
        def _safe_properties_count(element: dict) -> int:
            properties = element.get("Properties")
            return len(properties) if isinstance(properties, list) else 0

        def _safe_category_count(element: dict) -> int:
            # Keep consistent with the main helper: match the property-entry count.
            return _safe_properties_count(element)

        def _safe_distinct_category_count(element: dict) -> int:
            properties = element.get("Properties")
            if not isinstance(properties, list):
                return 0
            categories = set()
            for item in properties:
                if not isinstance(item, dict):
                    continue
                category = item.get("category")
                if category is None:
                    continue
                text = str(category).strip()
                if text:
                    categories.add(text)
            return len(categories)

        rows = []
        row_no = 0
        for element in data:
            if not isinstance(element, dict):
                continue
            name = element.get("Name")
            if name is None:
                continue
            row_no += 1
            rows.append({
                "No": row_no,
                "Name": name,
                "PropertiesCount": _safe_properties_count(element),
                "UniqueCategoryCount": _safe_distinct_category_count(element),
                "CategoryCount": _safe_category_count(element),
            })
    else:
        raise NameError("`rows` is not defined. Run Cell 5 (load JSON), Cell 6 (helpers), and Cell 7 (build rows), then run this cell again.")

# ---- Output options ----
preview_rows = None  # set to an integer (e.g., 20) to print only a preview
to_print = rows if preview_rows is None else rows[:preview_rows]

# Option A: aligned fixed-width output (better for notebook viewing than tabs)
headers = ["No", "Name", "PropertiesCount", "UniqueCategoryCount", "CategoryCount"]
numeric_cols = {"No", "PropertiesCount", "UniqueCategoryCount", "CategoryCount"}

# Compute widths (cap Name so the table doesn't get insanely wide)
max_name_width = 60
widths = {h: len(h) for h in headers}
for r in to_print:
    for h in headers:
        val = r.get(h, "")
        text = "" if val is None else str(val)
        if h == "Name" and len(text) > max_name_width:
            text = text[: max_name_width - 1] + "…"
        widths[h] = max(widths[h], len(text))
widths["Name"] = min(widths["Name"], max_name_width)

def _fmt_row(row: dict) -> str:
    parts = []
    for h in headers:
        val = row.get(h, "")
        text = "" if val is None else str(val)
        if h == "Name" and len(text) > widths[h]:
            text = text[: widths[h] - 1] + "…"
        align = ">" if h in numeric_cols else "<"
        parts.append(f"{text:{align}{widths[h]}}")
    return "  ".join(parts)

print("  ".join([f"{h:{('<' if h == 'Name' else '>')}{widths[h]}}" for h in headers]))
print("  ".join(["-" * widths[h] for h in headers]))
for r in to_print:
    print(_fmt_row(r))

if preview_rows is not None and len(rows) > preview_rows:
    print(f"\n(Preview only: showing {preview_rows} of {len(rows)} rows. Set preview_rows=None to show all.)")

# Option B: TSV file output (best for Excel / very large outputs)
output_tsv = input_path.with_suffix("").with_name(input_path.stem + "_counts.tsv")
with output_tsv.open("w", encoding="utf-8", newline="") as f:
    f.write("No\tName\tPropertiesCount\tUniqueCategoryCount\tCategoryCount\n")
    for r in rows:
        f.write(f"{r['No']}\t{r['Name']}\t{r['PropertiesCount']}\t{r['UniqueCategoryCount']}\t{r['CategoryCount']}\n")
print("Wrote:", output_tsv)

 No  Name                                                          PropertiesCount  UniqueCategoryCount  CategoryCount
---  ------------------------------------------------------------  ---------------  -------------------  -------------
  1  1JNL9322340_A-1JNL9322340 - Pyramid                                        46                    9             46
  2  1JNL9362899_A-Electrical design requirements, MVS1, NER, Au…               31                    3             31
  3  1JNL9441111_A-Cable Ladder 90 450                                          84                   11             84
  4  1JNL9441111_A-Cable Ladder 90 450                                         100                   11            100
  5  1JNL9441111_A-Cable Ladder 90 450                                         100                   11            100
  6  1JNL9442575_A-MVS Cable Ladder P1                                          30                    3             30
  7  1JNL9442576_A-K6 cable tray MVS P1         